# train_sarima.py
Trains and evaluates a fixed SARIMAX model for selected ZIP codes.

In [1]:
"""Train and evaluate a fixed SARIMAX model for selected ZIP codes.

Input: outputs/sarima_data.csv
Output: outputs/sarima_results.csv, outputs/sarima_forecasts.csv
"""
import warnings
from typing import Any
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
INPUT_FILE    = PROJECT_ROOT / "outputs" / "sarima_data.csv"
RESULTS_FILE  = PROJECT_ROOT / "outputs" / "sarima_results.csv"
FORECASTS_FILE = PROJECT_ROOT / "outputs" / "sarima_forecasts.csv"
ORDER          = (1, 1, 1)
SEASONAL_ORDER = (1, 1, 1, 12)
TRAIN_END      = "2022-12-31"
TEST_START     = "2023-01-31"
TARGET_ZIPS = {
    "02126": "Mattapan (boston_city)",
    "02116": "Back Bay (boston_city)",
    "02150": "Chelsea (inner_suburb)",
    "02139": "Cambridge (inner_suburb)",
    "01840": "Lawrence (outer_suburb)",
    "02481": "Wellesley (outer_suburb)",
}

In [2]:
def fit_sarimax(y, exog) -> Any:
    return SARIMAX(
        y, exog=exog,
        order=ORDER, seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False)

In [3]:
def main():
    df = pd.read_csv(INPUT_FILE, dtype={"zip_code": str})
    df["date"] = pd.to_datetime(df["date"])

    results = []
    forecasts = []

    for zip_code, label in TARGET_ZIPS.items():
        print(f"\n{zip_code} - {label}")

        sub = df[df["zip_code"] == zip_code].sort_values("date").set_index("date")
        y        = sub["zhvi"]

        train_y    = y[y.index <= TRAIN_END]
        test_y     = y[y.index >= TEST_START]
        train_exog = sub["MORTGAGE30US"][sub.index <= TRAIN_END].to_numpy(dtype=float).reshape(-1, 1)
        test_exog  = sub["MORTGAGE30US"][sub.index >= TEST_START].to_numpy(dtype=float).reshape(-1, 1)

        fitted: Any = fit_sarimax(train_y, train_exog)
        preds  = fitted.forecast(steps=len(test_y), exog=test_exog)

        mae  = mean_absolute_error(test_y, preds)
        rmse = np.sqrt(mean_squared_error(test_y, preds))
        mape = np.mean(np.abs((test_y.values - preds.values) / test_y.values)) * 100
        print(f"  MAE: ${mae:,.0f}  RMSE: ${rmse:,.0f}  MAPE: {mape:.1f}%")

        results.append({"zip_code": zip_code, "label": label,
                        "order": str(ORDER), "seasonal_order": str(SEASONAL_ORDER),
                        "mae": round(mae, 0), "rmse": round(rmse, 0),
                        "mape_pct": round(mape, 2)})

        forecast_df = pd.DataFrame({
            "zip_code": zip_code,
            "label": label,
            "date": test_y.index,
            "actual_zhvi": test_y.values,
            "pred_zhvi": preds.values,
        })
        forecasts.append(forecast_df)

    pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
    pd.concat(forecasts, ignore_index=True).to_csv(FORECASTS_FILE, index=False)
    print(f"\nResults -> {RESULTS_FILE}")
    print(f"Forecasts -> {FORECASTS_FILE}")

In [4]:
if __name__ == "__main__":
    main()


02126 - Mattapan (boston_city)
  MAE: $12,798  RMSE: $16,171  MAPE: 2.3%

02116 - Back Bay (boston_city)
  MAE: $35,860  RMSE: $45,115  MAPE: 2.8%

02150 - Chelsea (inner_suburb)
  MAE: $10,989  RMSE: $13,697  MAPE: 2.2%

02139 - Cambridge (inner_suburb)
  MAE: $26,581  RMSE: $34,105  MAPE: 2.9%

01840 - Lawrence (outer_suburb)
  MAE: $13,690  RMSE: $15,639  MAPE: 4.1%

02481 - Wellesley (outer_suburb)
  MAE: $33,014  RMSE: $40,427  MAPE: 1.8%

Results -> C:\Users\tanma\Desktop\DS4420\housingAffordability\outputs\sarima_results.csv
Forecasts -> C:\Users\tanma\Desktop\DS4420\housingAffordability\outputs\sarima_forecasts.csv
